# CTA historical service and ridership comparison

This notebook imports the eight dated CTA GTFS archives listed in the project manifest, aligns daily scheduled bus trips and trip-hours with CTA boardings by exact route and service date, then compares demand-only and service-aware regressors on the same matched rows.

Do not use the current google_transit.zip from CTA's schedule page for this historical comparison; it does not cover the ridership dates. Download each exact Mobility Database archive ID from the intern guide and put either the ZIP or its extracted folder in Google Drive under CTA_GTFS/<archive_id>/. ZIP files are checked against the guide SHA-256. For extracted folders, the notebook fingerprints the GTFS table contents but cannot verify the original ZIP bytes.

The notebook reports a retrospective development comparison through July, then evaluates a final fixed-origin August holdout. The August forecast is issued at the end of July, freezes the July 11 schedule archive, and feeds predictions, not actual August rides, into future demand lags. August labels are used only for scoring.

In [ ]:
from pathlib import Path
import subprocess
from google.colab import drive, files
drive.mount('/content/drive')
REPO = Path('/content/Powered-Public-Transit-Route-Optimization')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'users/bidhubhusan',
                    'https://github.com/springboardmentor441p-coderr/Powered-Public-Transit-Route-Optimization.git', str(REPO)], check=True)
%cd /content/Powered-Public-Transit-Route-Optimization
%pip -q install -r requirements.txt

## 1. Locate ridership and archived schedules

The ridership CSV downloads from the official portal if it is not already in the Colab session. Keep the eight schedule snapshots in separate folders under the Drive folder shown below.

In [ ]:
import pandas as pd
from urllib.request import urlretrieve
from src.gtfs_service import build_service_table
from src.compare_service_models import compare

RIDERSHIP_PATH = Path('/content/cta_bus_ridership.csv')
if not RIDERSHIP_PATH.exists():
    urlretrieve('https://data.cityofchicago.org/api/views/jyb9-n7fm/rows.csv?accessType=DOWNLOAD', RIDERSHIP_PATH)
GTFS_DRIVE_DIR = Path('/content/drive/MyDrive/CTA_GTFS')
manifest = pd.read_csv('data/raw/gtfs_archive_manifest.csv', dtype='string').fillna('')
def archive_location(archive_id):
    folder = GTFS_DRIVE_DIR / archive_id
    zipped = folder / 'original.zip'
    return str(zipped if zipped.is_file() else folder)
manifest['zip_path'] = manifest['archive_id'].map(archive_location)
MANIFEST_PATH = Path('/content/gtfs_archive_manifest_colab.csv')
manifest.to_csv(MANIFEST_PATH, index=False)
missing = [row.zip_path for row in manifest.itertuples() if not Path(row.zip_path).exists()]
if missing:
    raise FileNotFoundError('Place all eight ZIPs or extracted archive folders in Drive first. Missing:\n' + '\n'.join(missing))
print('Ridership file:', RIDERSHIP_PATH)
print('GTFS archive count:', len(manifest))

## 2. Import dated service and audit the exact join

The importer applies calendar_dates additions/removals, chooses the latest archive retrieved on or before each date for the whole network, and leaves unmatched rows visible. It also writes an August schedule using the latest archive available at the July 31 forecast origin. Stop-time tables are streamed, so this step can take several minutes.

In [ ]:
SERVICE_PATH = Path('data/processed/cta_route_date_demand_service.csv')
FIXED_SERVICE_PATH = Path('data/processed/cta_service_fixed_origin.csv')
IMPORT_REPORT = Path('outputs/reports/gtfs_service_import.json')
import_report = build_service_table(
    RIDERSHIP_PATH, MANIFEST_PATH, SERVICE_PATH, IMPORT_REPORT,
    fixed_origin_date='2026-07-31', fixed_start_date='2026-08-01',
    fixed_end_date='2026-08-31', fixed_output_path=FIXED_SERVICE_PATH)
print(pd.Series({k: v for k, v in import_report.items() if not isinstance(v, (dict, list))}))
print('Archive and date coverage details:')
display(pd.DataFrame(import_report['archive_versions']))
print('Unmatched ridership rows:', import_report['ridership_without_schedule_rows'])
print('Unmatched schedule rows:', import_report['schedule_without_ridership_rows'])

## 3. Compare demand-only and service-aware models

June-July are development dates. The fixed-origin August comparison fits through July 31, freezes the July 11 archive for every August date, and recursively uses model predictions after the July 31 origin. The evaluation code never uses actual August rides as lag features. Model settings stay unchanged from the earlier comparison; compare both fitted models with the same recursive past-four-same-weekdays baseline. Service-hour values remain missing when trip durations are incomplete; the pipeline imputes them from its training data and adds missingness indicators.

In [ ]:
COMPARISON_REPORT = Path('outputs/reports/service_model_comparison.json')
comparison = compare(
    RIDERSHIP_PATH, SERVICE_PATH, COMPARISON_REPORT, Path('models'),
    train_start='2025-11-23', train_end='2026-03-31',
    validation_start='2026-04-01', validation_end='2026-07-31',
    test_start='2026-08-01', test_end='2026-08-31',
    fixed_origin_service_path=FIXED_SERVICE_PATH, fixed_origin='2026-07-31')
print('August matched test coverage:', comparison['matched_test_coverage_pct'], '%')
print('Seasonal baselines on matched test rows:')
display(pd.DataFrame(comparison['seasonal_baselines_on_matched_test_rows']).T)
print('Model comparison:')
display(pd.DataFrame({name: result['test_metrics'] for name, result in comparison['models'].items()}).T)
print('Fixed-origin August results (primary holdout):')
display(pd.DataFrame({name: result['metrics'] for name, result in comparison['fixed_origin_evaluation']['models'].items()}).T)
print('Coverage, frozen schedule version, and detailed route/day-type errors are in the JSON report.')

## 4. Download the results

Save these reports with the project. The two fitted artifacts were trained on the same matched partitions and differ only in whether scheduled trips/service-hours are included.

In [ ]:
files.download(str(IMPORT_REPORT))
files.download(str(COMPARISON_REPORT))
files.download('models/cta_demand_only.joblib')
files.download('models/cta_service_aware.joblib')